In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def num_flat_features(self,x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [37]:
params = list(net.parameters())
print(len(params))
for param in params:
    print(param.size())

print(params[2])

10
torch.Size([6, 1, 5, 5])
torch.Size([6])
torch.Size([16, 6, 5, 5])
torch.Size([16])
torch.Size([120, 400])
torch.Size([120])
torch.Size([84, 120])
torch.Size([84])
torch.Size([10, 84])
torch.Size([10])
Parameter containing:
tensor([[[[-4.5492e-02, -3.3305e-02, -1.3111e-02, -2.8152e-02,  2.0160e-02],
          [-5.5733e-02, -1.5435e-02,  5.2404e-02,  4.7932e-02,  4.4575e-02],
          [ 2.0218e-02,  3.9784e-02, -2.2524e-03, -3.7035e-02,  2.5487e-02],
          [ 3.8502e-02, -6.7880e-02, -7.2675e-02, -4.9873e-03,  1.3242e-02],
          [ 1.0190e-02, -4.3457e-02, -4.6373e-02, -7.1521e-03,  5.7437e-02]],

         [[-3.0388e-02, -9.6065e-03,  5.7516e-02, -1.5912e-02,  2.4490e-02],
          [-7.7489e-02, -8.1544e-03, -4.8387e-02,  3.4051e-03, -3.2896e-03],
          [-3.4483e-02, -7.6398e-02, -5.5138e-03,  7.0576e-02,  4.7311e-03],
          [ 5.9375e-02, -5.4231e-02, -7.1332e-03,  6.7737e-02, -5.7956e-02],
          [-4.8640e-02, -1.0437e-02, -2.8746e-02,  7.2709e-03,  7.5239e-02]],


In [38]:
input = torch.randn(1, 1, 32, 32)
criterion = nn.MSELoss()
target = torch.randn(10)
target = target.view(1, -1)
print(target)

tensor([[ 4.3234e-01, -6.0185e-02,  1.2576e-01,  1.0109e+00,  7.1336e-01,
          1.7593e+00,  7.9914e-02, -1.1008e-03,  3.4676e-01,  9.5120e-01]])


In [39]:
import torch.optim as optim

optimizer = optim.SGD(net.parameters(), lr=0.1)

optimizer.zero_grad()
output = net(input)
loss = criterion(output, target)
print(loss)
print(loss.shape)

tensor(0.5570, grad_fn=<MseLossBackward0>)
torch.Size([])


In [40]:
net.zero_grad()

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)
print(net.conv2.bias.grad)

loss.backward()
print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
None
conv1.bias.grad before backward
tensor([ 0.0056, -0.0107, -0.0043,  0.0015, -0.0015, -0.0009])


In [41]:
optimizer.step()